# Fetch Isochrones Pipelines

Fetches isochrone polygons from the Mapbox API for each library branch.
Three pipelines, one per mode:
- **driving** — 5 min
- **cycling** — 10 min
- **walking** — 15 min

Caches results in Feast with a 1-hour freshness window.
Preserves historical isochrone data to track changes over time.

Pipelines run sequentially to stay within the Mapbox rate limit.

In [ ]:
import os
import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

## Component

In [ ]:
RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"


@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "requests", "shapely", "pandas", "pyarrow", "feast"],
    pip_index_urls=["https://pypi.org/simple"],
)
def fetch_isochrones(
    mode: str,
    travel_time: int,
    isochrones_out: Output[Dataset],
):
    import io
    import os
    import time
    import logging
    import requests
    import pandas as pd
    import boto3
    from datetime import datetime, timezone, timedelta
    from shapely.geometry import shape

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("fetch_isochrones")

    RATE_LIMIT_DELAY = 0.25
    MAX_RETRIES = 3
    FRESHNESS_HOURS = 1

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]
    mapbox_token = os.environ["MAPBOX_TOKEN"]

    # --- Feast setup ---
    from feast import FeatureStore, Entity, FeatureView, Field
    from feast.data_source import PushSource
    from feast.types import String, Int64, Float64

    feast_config = os.environ.get("FEAST_CONFIG_DIR", "/feast-config")
    store = FeatureStore(repo_path=feast_config)

    isochrone_entity = Entity(
        name="isochrone",
        join_keys=["branch", "mode", "travel_time"],
    )

    isochrone_push = PushSource(
        name="isochrone_push",
        schema=[
            Field(name="branch", dtype=String),
            Field(name="mode", dtype=String),
            Field(name="travel_time", dtype=Int64),
            Field(name="latitude", dtype=Float64),
            Field(name="longitude", dtype=Float64),
            Field(name="geometry_wkt", dtype=String),
            Field(name="fetch_ts", dtype=Int64),
        ],
    )

    isochrone_fv = FeatureView(
        name="isochrone_features",
        entities=[isochrone_entity],
        schema=[
            Field(name="latitude", dtype=Float64),
            Field(name="longitude", dtype=Float64),
            Field(name="geometry_wkt", dtype=String),
            Field(name="fetch_ts", dtype=Int64),
        ],
        source=isochrone_push,
        online=True,
        ttl=timedelta(0),
    )

    store.apply([isochrone_entity, isochrone_fv, isochrone_push])
    logger.info("Feast feature definitions applied")

    # --- Load branches ---
    obj = s3.get_object(Bucket=bucket, Key="clean/branches.csv")
    branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
    branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
    logger.info(f"Loaded {len(branches)} active branches")

    now = datetime.now(timezone.utc)
    freshness_cutoff = now - timedelta(hours=FRESHNESS_HOURS)
    cutoff_ts = int(freshness_cutoff.timestamp())
    base_url = f"https://api.mapbox.com/isochrone/v1/mapbox/{mode}/"

    new_rows = []
    stats = {"cached": 0, "fetched": 0, "failed": 0}

    for _, branch in branches.iterrows():
        branch_name = branch["BRANCH"]
        lat, lon = branch["LATITUDE"], branch["LONGITUDE"]

        try:
            feast_result = store.get_online_features(
                features=["isochrone_features:fetch_ts"],
                entity_rows=[{
                    "branch": branch_name,
                    "mode": mode,
                    "travel_time": travel_time,
                }],
            ).to_dict()
            cached_ts = feast_result.get("fetch_ts", [None])[0]
            if cached_ts is not None and cached_ts >= cutoff_ts:
                stats["cached"] += 1
                continue
        except Exception as e:
            logger.debug(f"Feast lookup failed for {branch_name}: {e}")

        success = False
        for attempt in range(MAX_RETRIES):
            try:
                url = (
                    f"{base_url}{lon},{lat}"
                    f"?contours_minutes={travel_time}"
                    f"&polygons=true&access_token={mapbox_token}"
                )
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                geometry = resp.json()["features"][0]["geometry"]
                geometry_wkt = shape(geometry).wkt

                new_rows.append({
                    "branch": branch_name,
                    "mode": mode,
                    "travel_time": travel_time,
                    "latitude": lat,
                    "longitude": lon,
                    "geometry_wkt": geometry_wkt,
                    "fetch_ts": int(now.timestamp()),
                    "event_timestamp": now,
                })
                stats["fetched"] += 1
                success = True
                break
            except Exception as e:
                logger.warning(
                    f"Attempt {attempt + 1}/{MAX_RETRIES} failed for "
                    f"{branch_name} ({mode}, {travel_time}min): {e}"
                )
                if attempt < MAX_RETRIES - 1:
                    time.sleep(1)

        if not success:
            stats["failed"] += 1
            logger.error(
                f"All {MAX_RETRIES} attempts failed for "
                f"{branch_name} ({mode}, {travel_time}min)"
            )

        time.sleep(RATE_LIMIT_DELAY)

    if new_rows:
        push_df = pd.DataFrame(new_rows)
        push_df["event_timestamp"] = pd.to_datetime(push_df["event_timestamp"], utc=True)
        store.push("isochrone_push", push_df, to=PushSource.PushMode.ONLINE)
        logger.info(f"Pushed {len(push_df)} isochrones to Feast")

    cache_key = "features/isochrones.parquet"
    existing = pd.DataFrame()
    try:
        obj = s3.get_object(Bucket=bucket, Key=cache_key)
        existing = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    except Exception:
        pass

    if new_rows:
        new_df = pd.DataFrame(new_rows)
        result = pd.concat([existing, new_df], ignore_index=True) if not existing.empty else new_df
    else:
        result = existing if not existing.empty else pd.DataFrame()

    if not result.empty:
        buf = io.BytesIO()
        result.to_parquet(buf, index=False)
        buf.seek(0)
        s3.put_object(Bucket=bucket, Key=cache_key, Body=buf.getvalue())

    result.to_parquet(isochrones_out.path, index=False)

    logger.info(
        f"Done ({mode}, {travel_time}min). "
        f"Cached: {stats['cached']}, Fetched: {stats['fetched']}, "
        f"Failed: {stats['failed']}. Total rows: {len(result)}"
    )

## Pipeline Definitions

In [ ]:
def _mount_secrets(task):
    kubernetes.use_secret_as_env(
        task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )
    kubernetes.use_secret_as_env(
        task,
        secret_name="mapbox-token",
        secret_key_to_env={"MAPBOX_TOKEN": "MAPBOX_TOKEN"},
    )
    kubernetes.use_config_map_as_volume(
        task,
        config_map_name="feast-cpl-features-client",
        mount_path="/feast-config",
    )
    kubernetes.use_config_map_as_volume(
        task,
        config_map_name="feast-cpl-features-client-ca",
        mount_path="/etc/pki/tls/custom-certs",
    )


@dsl.pipeline(
    name="cpl-fetch-isochrones-driving",
    description="Fetch driving isochrones (5 min) for CPL branches",
)
def fetch_driving_pipeline():
    task = fetch_isochrones(mode="driving", travel_time=5)
    _mount_secrets(task)


@dsl.pipeline(
    name="cpl-fetch-isochrones-cycling",
    description="Fetch cycling isochrones (10 min) for CPL branches",
)
def fetch_cycling_pipeline():
    task = fetch_isochrones(mode="cycling", travel_time=10)
    _mount_secrets(task)


@dsl.pipeline(
    name="cpl-fetch-isochrones-walking",
    description="Fetch walking isochrones (15 min) for CPL branches",
)
def fetch_walking_pipeline():
    task = fetch_isochrones(mode="walking", travel_time=15)
    _mount_secrets(task)

## Submit

Registers all three pipeline definitions, then runs them sequentially
to stay within the Mapbox API rate limit (300 req/min).

In [ ]:
import subprocess
import time as _time
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

token = subprocess.check_output("oc whoami -t", shell=True, text=True).strip()
pipeline_url = os.environ.get("PIPELINES_URL")

client = kfp.Client(
    host=pipeline_url,
    existing_token=token,
    verify_ssl=False,
)

PIPELINES = [
    ("cpl-fetch-isochrones-driving", fetch_driving_pipeline),
    ("cpl-fetch-isochrones-cycling", fetch_cycling_pipeline),
    ("cpl-fetch-isochrones-walking", fetch_walking_pipeline),
]

# Register all pipeline definitions
for name, pipeline_func in PIPELINES:
    yaml_file = f"{name}.yaml"
    Compiler().compile(pipeline_func, yaml_file)
    try:
        client.upload_pipeline(yaml_file, pipeline_name=name)
        print(f"{name}: registered.")
    except ApiException as e:
        if e.status == 409:
            print(f"{name}: already exists.")
        else:
            raise

# Run sequentially — wait for each to finish before starting the next
for name, pipeline_func in PIPELINES:
    print(f"\nStarting {name}...")
    run = client.create_run_from_pipeline_func(
        pipeline_func,
        experiment_name="cpl-coverage",
        run_name=name,
        enable_caching=False,
    )
    print(f"  submitted (run_id={run.run_id})")

    while True:
        status = client.get_run(run.run_id).state
        if status in ("SUCCEEDED", "FAILED", "SKIPPED", "ERROR"):
            break
        _time.sleep(15)

    print(f"  finished: {status}")
    if status != "SUCCEEDED":
        print(f"  WARNING: {name} did not succeed, continuing anyway...")

print("\nAll pipelines complete.")